# SBS validation: OMTPL Retail Classical Frequency

Ноутбук запускает полный набор тестов SBS для `OMTPL_Retail_Classical_FREQ_Inference_db_ver01.csv`.

Основная настройка:
- модель: frequency-регрессия;
- score: `GLM_FREQ`, то есть чистое предсказание частоты;
- exposure/weight: `Exposure`;
- `GLM_FREQ_weighted = GLM_FREQ * Exposure` проверяется как контрольная производная колонка, но не подается как основной score;
- временная колонка: один из `TIME_TREND_ID_START_QUARTER`, `TIME_TREND_SEAS_REPORT_YEAR_QUARTER`;
- prediction bins `GLM_MODEL_BINQ10`, `GLM_MODEL_weighted_BINQ10` исключаются из признаков, потому что это производные от предсказания модели.


In [ ]:
from pathlib import Path
import sys
import warnings

import pandas as pd
import numpy as np
from IPython.display import HTML, display

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 160)
pd.set_option("display.max_rows", 160)

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name.lower() == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

DATA_PATH = PROJECT_ROOT / "data" / "raw" / "OMTPL_Retail_Classical_FREQ_Inference_db_ver01.csv"
REPORTS_DIR = PROJECT_ROOT / "reports" / "omtpl_retail_classical_freq"

print(f"PROJECT_ROOT: {PROJECT_ROOT}")
print(f"DATA_PATH: {DATA_PATH}")
print(f"REPORTS_DIR: {REPORTS_DIR}")


In [ ]:
def read_csv_robust(path: Path) -> pd.DataFrame:
    if not path.exists():
        raise FileNotFoundError(f"CSV не найден: {path}")

    errors = []
    for encoding in ["utf-8-sig", "cp1251", "windows-1251", "latin1"]:
        try:
            return pd.read_csv(path, sep=None, engine="python", encoding=encoding)
        except Exception as exc:
            errors.append(f"{encoding}: {exc}")

    raise ValueError("Не удалось прочитать CSV:\n" + "\n".join(errors))


data = read_csv_robust(DATA_PATH)
data.columns = [str(c).strip() for c in data.columns]

print(data.shape)
display(data.head())
display(pd.DataFrame({"column": data.columns, "dtype": [str(data[c].dtype) for c in data.columns]}))


## Настройки колонок

Если автоопределение выбрало не те поля, переопредели значения в следующей ячейке вручную.

In [ ]:
ID_CANDIDATES = ["UNIQUE_KEY", "ID", "IDpol", "POLICY_ID", "CONTRACT_ID", "AGREEMENT_ID"]
SAMPLE_CANDIDATES = ["DB_SPLIT", "sample", "SAMPLE", "Sample"]
TARGET_CANDIDATES = ["ClaimsCountPaid", "ClaimsCount", "ClaimNb", "DAGO_ClaimsCount"]
WEIGHT_CANDIDATES = ["Exposure", "EXPOSURE", "exposure"]
DATE_CANDIDATES = ["TIME_TREND_ID_START_QUARTER", "TIME_TREND_SEAS_REPORT_YEAR_QUARTER"]

SCORE_COLUMN = "GLM_FREQ"
WEIGHTED_SCORE_COLUMN = "GLM_FREQ_weighted"
VALIDATION_MODE = "oos"  # "oos" ожидает TEST/OOS в split; "oot" ожидает OOT

PREDICTION_DERIVED_COLUMNS = {
    "GLM_FREQ_weighted",
    "GLM_MODEL_BINQ10",
    "GLM_MODEL_weighted_BINQ10",
    "GLM_MODEL_BIN_Q10",
    "GLM_MODEL_WEIGHTED_BIN_Q10",
}


def first_existing(columns, candidates, field_name):
    for candidate in candidates:
        if candidate in columns:
            return candidate
    raise KeyError(f"Не удалось определить {field_name}. Кандидаты: {candidates}")


ID_COLUMN = first_existing(data.columns, ID_CANDIDATES, "id_column")
SAMPLE_COLUMN = first_existing(data.columns, SAMPLE_CANDIDATES, "sample_column")
DATE_COLUMN = first_existing(data.columns, DATE_CANDIDATES, "date_column")
TARGET_COLUMN = first_existing(data.columns, TARGET_CANDIDATES, "target_column")
WEIGHT_COLUMN = first_existing(data.columns, WEIGHT_CANDIDATES, "weight_column")

required = [ID_COLUMN, SAMPLE_COLUMN, DATE_COLUMN, TARGET_COLUMN, SCORE_COLUMN, WEIGHT_COLUMN]
missing = [c for c in required if c not in data.columns]
if missing:
    raise KeyError(f"В данных отсутствуют обязательные колонки: {missing}")

pd.DataFrame({
    "role": ["id", "sample", "date", "target", "score", "weight"],
    "column": [ID_COLUMN, SAMPLE_COLUMN, DATE_COLUMN, TARGET_COLUMN, SCORE_COLUMN, WEIGHT_COLUMN],
})


## Подготовка данных под контракт SBS

Для frequency-модели в SBS подается `score_column = GLM_FREQ` и `weight_column = Exposure`: внутри M0/M2/M4/M5 библиотека сама использует экспозицию для взвешенных метрик и калибровки.

In [ ]:
def to_numeric_series(series: pd.Series, column: str) -> pd.Series:
    if pd.api.types.is_numeric_dtype(series):
        return pd.to_numeric(series, errors="raise")

    cleaned = (
        series.astype(str)
        .str.strip()
        .str.replace("\u00a0", "", regex=False)
        .str.replace(" ", "", regex=False)
        .str.replace(",", ".", regex=False)
    )
    cleaned = cleaned.mask(cleaned.str.lower().isin({"", "nan", "none", "null"}))
    return pd.to_numeric(cleaned, errors="raise")


final_df = data.copy()

final_df[ID_COLUMN] = final_df[ID_COLUMN].astype(str)
final_df[SAMPLE_COLUMN] = final_df[SAMPLE_COLUMN].astype(str).str.strip().str.upper()
final_df[DATE_COLUMN] = pd.to_datetime(final_df[DATE_COLUMN], errors="raise", dayfirst=True)

for column in [TARGET_COLUMN, SCORE_COLUMN, WEIGHT_COLUMN, WEIGHTED_SCORE_COLUMN]:
    if column in final_df.columns:
        final_df[column] = to_numeric_series(final_df[column], column)

if (final_df[WEIGHT_COLUMN] <= 0).any():
    bad_count = int((final_df[WEIGHT_COLUMN] <= 0).sum())
    raise ValueError(f"В {WEIGHT_COLUMN} есть неположительные значения: {bad_count}")

if WEIGHTED_SCORE_COLUMN in final_df.columns:
    expected_weighted = final_df[SCORE_COLUMN] * final_df[WEIGHT_COLUMN]
    max_abs_diff = float((final_df[WEIGHTED_SCORE_COLUMN] - expected_weighted).abs().max())
    print(f"Check {WEIGHTED_SCORE_COLUMN} ~= {SCORE_COLUMN} * {WEIGHT_COLUMN}: max_abs_diff={max_abs_diff:.8g}")

service_columns = {
    ID_COLUMN,
    SAMPLE_COLUMN,
    DATE_COLUMN,
    TARGET_COLUMN,
    SCORE_COLUMN,
    WEIGHT_COLUMN,
    "ClaimsAmountPaid",
    "INTERCEPT",
    "INTERCEPT_VALUE",
    *PREDICTION_DERIVED_COLUMNS,
}

numeric_features = [
    c for c in final_df.columns
    if c.endswith("_SCORING") and c not in service_columns
]

for column in numeric_features:
    final_df[column] = to_numeric_series(final_df[column], column)

if not numeric_features:
    numeric_features = [
        c for c in final_df.columns
        if c not in service_columns and pd.api.types.is_numeric_dtype(final_df[c])
    ]

# По умолчанию второй TIME_TREND не включаем в фичи, чтобы не смешивать дату тестов с фактором.
# Если нужно протестировать второй time factor как категориальный признак, раскомментируй блок ниже.
categorical_features = []
# categorical_features = [
#     c for c in DATE_CANDIDATES
#     if c in final_df.columns and c != DATE_COLUMN and c not in service_columns
# ]

print(f"numeric_features: {len(numeric_features)}")
print(f"categorical_features: {len(categorical_features)}")
display(pd.DataFrame({"numeric_feature": pd.Series(numeric_features)}))


In [ ]:
split_summary = final_df[SAMPLE_COLUMN].value_counts(dropna=False).rename_axis(SAMPLE_COLUMN).reset_index(name="rows")
date_summary = pd.DataFrame({
    "metric": ["min_date", "max_date", "missing_dates"],
    "value": [final_df[DATE_COLUMN].min(), final_df[DATE_COLUMN].max(), final_df[DATE_COLUMN].isna().sum()],
})
score_columns_to_show = [c for c in [TARGET_COLUMN, SCORE_COLUMN, WEIGHT_COLUMN, WEIGHTED_SCORE_COLUMN, "GLM_MODEL_BINQ10", "GLM_MODEL_weighted_BINQ10"] if c in final_df.columns]

display(split_summary)
display(date_summary)
display(final_df[score_columns_to_show].describe(include="all").T)

feature_quality = pd.DataFrame({
    "feature": numeric_features,
    "missing": [int(final_df[c].isna().sum()) for c in numeric_features],
    "nunique": [int(final_df[c].nunique(dropna=True)) for c in numeric_features],
    "min": [final_df[c].min() for c in numeric_features],
    "max": [final_df[c].max() for c in numeric_features],
})
display(feature_quality)


## Конфиг SBS

In [ ]:
config = {
    "task": "regression",
    "model_mode": "frequency",
    "validation_mode": VALIDATION_MODE,
    "columns": {
        "numeric_features": numeric_features,
        "categorical_features": categorical_features,
        "score_column": SCORE_COLUMN,
        "date_column": DATE_COLUMN,
        "target_column": TARGET_COLUMN,
        "sample_column": SAMPLE_COLUMN,
        "id_column": ID_COLUMN,
        "weight_column": WEIGHT_COLUMN,
    },
}

config


In [ ]:
from sbs_evaluation_tool.core.config_validation import validate_config

validated_config = validate_config(config, final_df)
print(validated_config)
print(f"Numeric features: {len(validated_config.columns.numeric_features)}")
print(f"Categorical features: {len(validated_config.columns.categorical_features)}")


## Быстрый inline smoke: M2.1 Gini

Эта ячейка полезна перед полным прогоном: быстро проверяет, что target, score, split и exposure совместимы с ranking-тестом.

In [ ]:
from sbs_evaluation_tool.core.inline_display import show_test_inline
from sbs_evaluation_tool.tests.m2_gini_tests import M21_GiniTest

show_test_inline(M21_GiniTest(), {"df": final_df, "config": config})


## Полный запуск M0-M5 и генерация HTML-отчета

`RUN_SIZE = "normal"` запускает все группы, включая M3; `RUN_SIZE = "heavy"` пропускает M3 в текущем runner.

In [ ]:
from sbs_evaluation_tool.core.runner import run_sbs_evaluation

RUN_SIZE = "normal"  # "normal" = полный запуск; "heavy" = пропустить M3

report_path = run_sbs_evaluation(
    data=final_df,
    config=config,
    segment_name="OMTPL_Retail_Classical_FREQ",
    output_dir=str(REPORTS_DIR),
    size=RUN_SIZE,
)

print(report_path)


In [ ]:
report_file = Path(report_path).resolve()
display(HTML(f'<a href="{report_file.as_uri()}" target="_blank">Open SBS HTML report</a>'))
